Partendo dal modello generato nella lezione
Il modello è stato progettato per generare testo, ma la sua personalità dipende dal parametro temperature.
Sperimentazione: utilizzare il codice della lezione, generare tre testi di 400 caratteri pertando dallo stesso seed 'KING LEAR'
Usa temperatura = 0.1 (approccio convervativo)
Usa temperatura = 1.0 (bilanciamento standard)
Usa temperatura = 2.0 (approccio caotico)

Analisi: osserva i risultati e identifica in quale dei tre casi il modello inizia a inventare parole inesistenti o a ignorare completamente la grammatica inglese
Modifica: Aggiungi un tezo layer GRU intermedio al modello per aumentare la capacità di apprendimento

In [ ]:
import os
import numpy as np
import tensorflow as tf
import keras
import torch  # Necessario per la gestione della memoria torch.no_grad()

# --- ONFIGURAZIONE BACKEND E PERFORMANCE ---
os.environ["KERAS_BACKEND"] = "torch"

# Utilizzo di precisione mista a 16-bit per ottimizzare le prestazioni su GPU moderne (es. 7900 XTX)
keras.mixed_precision.set_global_policy("mixed_float16")

# --- ACQUISIZIONE E PRE-PROCESSAMENTO DEI DATI ---
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt', 
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
vocab = sorted(set(text))
char2idx = {u: i for i, u in enumerate(vocab)}
idx2char = np.array(vocab)

text_as_int = np.array([char2idx[c] for c in text])

# --- CREAZIONE DEL DATASET OTTIMIZZATO ---
seq_length = 250
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)

BATCH_SIZE = 128
BUFFER_SIZE = 10000
dataset = dataset.cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

# --- DEFINIZIONE DEL MODELLO (CON 3 LAYER GRU) ---
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    model = keras.Sequential([
        # Embedding Layer
        keras.layers.Embedding(vocab_size, embedding_dim),
        
        # 1° Layer GRU
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False, recurrent_initializer='glorot_uniform'),
        
        # 2° Layer GRU (Intermedio - Modifica richiesta)
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        
        # 3° Layer GRU (Aggiunto per aumentare la capacità di apprendimento)
        keras.layers.GRU(rnn_units, return_sequences=True, stateful=False),
        
        # Dense Layer (Logits)
        keras.layers.Dense(vocab_size)
    ])
    return model

model = build_model(vocab_size, embedding_dim, rnn_units, BATCH_SIZE)

# --- TRAINING ---
model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

# Nota: Epochs impostate a 50 come richiesto; ridurre per test rapidi
model.fit(dataset, epochs=50) 


# --- FUNZIONE DI GENERAZIONE E ANALISI DELLA TEMPERATURA ---
def generate_text(model, start_string, temperature=0.7, num_generate=400):
    input_eval = [char2idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []

    # Disattiviamo i gradienti per risparmiare memoria durante l'inferenza
    with torch.no_grad():
        for i in range(num_generate):
            predictions = model(input_eval)
            # Applichiamo la temperatura ai logits
            predictions = predictions[:, -1, :] / temperature
            
            predicted_id_tensor = keras.random.categorical(predictions, num_samples=1)
            predicted_id = int(keras.ops.convert_to_numpy(predicted_id_tensor)[0, 0])

            new_char_tensor = tf.expand_dims([predicted_id], 0)
            input_eval = tf.concat([input_eval, new_char_tensor], axis=-1)
            
            # Sliding window per mantenere il contesto
            if input_eval.shape[1] > seq_length:
                input_eval = input_eval[:, 1:]

            text_generated.append(idx2char[predicted_id])

    return (start_string + ''.join(text_generated))

# --- LOOP DI SPERIMENTAZIONE ---
temperatures = [0.1, 1.0, 2.0]
seed = "KING LEAR: "

print(f"\n{'='*50}")
print(f"ANALISI DELLA PERSONALITÀ DEL MODELLO (Seed: {seed})")
print(f"{'='*50}\n")

for temp in temperatures:
    print(f"--- TEST CON TEMPERATURA: {temp} ---")
    generated = generate_text(model, start_string=seed, temperature=temp, num_generate=400)
    print(generated)
    print(f"\n{'-'*50}\n")